In [32]:
import numpy as np
import matplotlib.pyplot as plt
import simulate_data_script as sim
from importlib import reload 
reload(sim)

n = 1000
d = 100
k = 50
corr = 0.9

pi0 = 0.05
snr = 1

rng = np.random.default_rng(42)

X = sim.X_design(n, d, "corr_blocks", k = k, corr = corr, intercept=True, rng=rng)

n, d_total = X.shape 

beta, noise_var = sim.beta_sparse(d, X, pi0, snr, rng=rng)
beta = np.concatenate([[0], beta])
beta_blocks = np.concatenate([[0], 1 + np.arange(d) // k]) 
    
y_latent, l, u = sim.y_tobit(X, beta, 20, 80, noise_var, rng)

y = np.clip(y_latent, l, u)

true_tau2 = beta[beta!=0].var()

In [34]:
G = X.T@X

rho_true = (beta != 0).astype(int)
P = np.diag(rho_true)
I = np.eye(len(rho_true))
R = G * (P @ (I - P) + np.outer(rho_true, rho_true))

S = np.linalg.inv(1/noise_var * R + 1/true_tau2 * np.eye(R.shape[0])) 
m = 1/noise_var *S@P @X.T @ y_latent

tau2_em = (np.trace(S) + m.T@m)/d

print(f"True tau2 {true_tau2}, tau2 EM: {tau2_em}")

True tau2 0.1686310541926808, tau2 EM: 0.17322007619434754
